# 13. The left hand, and the boundary

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

The right-hand file cannot stand in for the left hand by flipping blend shapes yourself. The left hand is a separate release.


## Loading the left hand


In [ ]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

from official_mano import DEFAULT_LEFT, OfficialMano

print(DEFAULT_LEFT)
left = OfficialMano(DEFAULT_LEFT)
out = left.forward(flat_hand=True)
print("left vertices", out["vertices"].shape, "joints", out["joints"].shape)
print("left wrist -> middle MCP", np.round(out["joints"][4] - out["joints"][0], 4))


/Users/alexlee/ZJUI26Fall/ST/Rebuild2hand/Forceband/MANO/mano_v1_2/models/MANO_LEFT.pkl
left vertices (778, 3) joints (16, 3)
left wrist -> middle MCP [ 0.0947 -0.0015 -0.0034]


`DEFAULT_LEFT` points at `mano_v1_2/models/MANO_LEFT.pkl`. The template, the shape basis, and the pose basis in that file are already mirrored. The X component from the left wrist to the middle MCP is positive, the opposite of the right hand's -X.

## The sign rule when you mirror a pose only

If you already have a right-hand axis-angle and you want the rotation mirrored across the YZ plane (`x → -x`), change every joint like this:

- `rx` stays
- `ry` changes sign
- `rz` changes sign

The `x` coordinate of every point changes sign. If you build triangles yourself, also swap the winding of each face so the normals point outward.

Let `S = diag(-1, 1, 1)`. The mirrored rotation matrix is `S @ R @ S`. It equals the rotation matrix of the axis-angle after `ry` and `rz` have changed sign.


In [ ]:
import mano_skeleton as sk

axis_angle = np.array([0.4, -0.7, 0.2])
mirrored = axis_angle.copy()
mirrored[1] *= -1
mirrored[2] *= -1
reflect = np.diag([-1.0, 1.0, 1.0])
by_matrix = reflect @ sk.rodrigues(axis_angle) @ reflect
by_axis = sk.rodrigues(mirrored)
print(np.round(by_matrix, 4))
print("the two constructions match?", np.allclose(by_matrix, by_axis))
assert np.allclose(by_matrix, by_axis)


[[ 0.7499  0.3099  0.5845]
 [-0.0456  0.9056 -0.4216]
 [-0.66    0.2895  0.6933]]
the two constructions match? True


The mesh itself still comes from `MANO_LEFT.pkl`. Mirroring only the axis-angles and then multiplying by the right hand's `posedirs` applies the right hand's flesh correction, and the surface will not match.

## What the model does not contain

- There is no 17th joint. A fingertip is not a hinge.
- The wrist is one joint. There is no row of separate carpals.
- The pose blend is linear in `R - I`. Far from the training poses the surface becomes implausible.
- `betas` covers the body-shape variation in the training scans. Past about ±2, a coefficient quickly stops looking like a real hand. A fit usually penalizes large `betas`.

That boundary is the question MANO can answer: given shape and pose, where are the mesh and the 16 joints. It does not detect a hand in an image, and it does not give the fingertip its own degree of freedom.

The next chapter turns "I have measured some points and I want the pose back" into an optimization of this same forward formula.
